# Текущий контроль 5–6. Метод k ближайших соседей
## Определение качества красного вина
Выбор k проводится только по тренировочной выборке. Масштабирование включено в Pipeline, поэтому тестовые данные не участвуют в обучении преобразований.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.metrics import mean_absolute_error
wine=pd.read_csv('winequality-red.csv')
display(wine.head());print('Размер:',wine.shape,'Пропуски:',wine.isna().sum().sum())
display(wine.quality.value_counts().sort_index())
wine.quality.value_counts().sort_index().plot.bar(title='Распределение качества вина');plt.show()
X=wine.drop(columns='quality');y=wine.quality
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
search=GridSearchCV(Pipeline([('scale',StandardScaler()),('knn',KNeighborsClassifier())]),{'knn__n_neighbors':[3,5,7,9,11,15,21],'knn__weights':['uniform','distance']},cv=StratifiedKFold(5,shuffle=True,random_state=42),scoring='f1_macro',n_jobs=1)
search.fit(Xtr,ytr);pred=search.predict(Xte)
print('Лучшие параметры:',search.best_params_,'CV macro F1:',search.best_score_)
print('Тест accuracy:',accuracy_score(yte,pred),'macro F1:',f1_score(yte,pred,average='macro'))
print('Базовая accuracy:',DummyClassifier(strategy='most_frequent').fit(Xtr,ytr).score(Xte,yte))
print('Средняя абсолютная ошибка:',mean_absolute_error(yte,pred))
print(classification_report(yte,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(yte,pred,labels=sorted(y.unique()),cmap='Blues');plt.title('Качество вина: тестовая выборка');plt.show()
pd.DataFrame({'actual':yte,'predicted':pred}).to_csv('wine_predictions.csv',index_label='row')
print(f'Вывод: accuracy={accuracy_score(yte,pred):.3f}, macro F1={f1_score(yte,pred,average="macro"):.3f}. Близкие оценки качества смешиваются. Для редких классов недостаточно примеров. Масштабирование необходимо для корректного сравнения расстояний.')


Оценка получена на отложенных 30% данных. Исходный набор содержит повторяющиеся строки, поэтому случайное разбиение может давать оптимистичную оценку для новых партий вина. Для такой задачи желательно разделять данные по партиям или удалять дубликаты перед разбиением.